In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

In [ ]:
protein_interaction[protein_interaction['Translated_protein_2'] == '']

In [ ]:
# ppi_available_genes = list(set(protein_interaction['Translated_protein_1']).union(set(protein_interaction['Translated_protein_2'])))

In [ ]:
# len(set(ppi_available_genes))

In [ ]:
# protein_working = protein_interaction[protein_interaction['combined_score']>700]
# protein_working[protein_working['Translated_protein_1'] == 'EGFR'][protein_interaction['Translated_protein_2'] =='SOX2']

### DrugBank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        
        # Extract toxicity information at drug level
        toxicity = drug.find('db:toxicity', ns)
        toxicity_text = toxicity.text if toxicity is not None else None
        
        # Extract categories (may contain toxicity classifications)
        categories = drug.findall('db:categories/db:category', ns)
        category_list = []
        for cat in categories:
            cat_name = cat.find('db:category', ns)
            if cat_name is not None:
                category_list.append(cat_name.text)
        categories_str = ', '.join(category_list) if category_list else None
        
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None,
                    'toxicity': toxicity_text,
                    'categories': categories_str
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None,
                        'toxicity': toxicity_text,
                        'categories': categories_str
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

### Genetic results

In [ ]:
### import data
# ### genes
# hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
# hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

# hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
# hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
import pandas as pd
### genes
### import data
### genes
hpv_positive_amplification_top_gene_df= pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV positive amplification top genes.csv')
hpv_positive_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_positive_deletion_top_gene_df= pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV positive deletion top genes.csv')
hpv_positive_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_positive_deletion_top_gene_df['amplification_count'] = hpv_positive_deletion_top_gene_df['deletion_count']
hpv_positive_genes = pd.concat([hpv_positive_amplification_top_gene_df, hpv_positive_deletion_top_gene_df], axis=0)
hpv_positive_genes =hpv_positive_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_negative_amplification_top_gene_df = pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV negative amplification top genes.csv')
hpv_negative_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_deletion_top_gene_df = pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV negative deletion top genes.csv')
hpv_negative_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_negative_deletion_top_gene_df['amplification_count'] = hpv_negative_deletion_top_gene_df['deletion_count']
hpv_negative_genes = pd.concat([hpv_negative_amplification_top_gene_df, hpv_negative_deletion_top_gene_df], axis=0)
hpv_negative_genes =hpv_negative_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('../2. Genetic based drug repurposing/Results/SOM results/HPV positive top genes.csv')
hpv_negative_som_genes = pd.read_csv('../2. Genetic based drug repurposing/Results/SOM results/HPV negative top genes.csv')


### Literature results

In [ ]:
extracted_target_df= pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
scraped_lit_data = pd.read_csv('../3. Literature based validation/Data/extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv', on_bad_lines='skip')
extracted_target_df_combined = pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()

In [ ]:
key_genes = ['PIK3CA', 'TP53', 'CDKN2A','SOX2', 'EGFR', 'HRAS', 'NOTCH1', 'FAT1', 'CASP8', 'KMT2D']
key_genes = [gene.lower() for gene in key_genes]
extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]
key_genes_indexes = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)].index.tolist()
key_genes_PMID = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]['PMID'].unique().tolist()

In [ ]:
key_genes = ['SOX2', 'TP53']
key_genes = [gene.lower() for gene in key_genes]
extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]
key_genes_indexes = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)].index.tolist()
key_genes_PMID = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]['PMID'].unique().tolist()

In [ ]:
scraped_lit_data[scraped_lit_data['PMID'].isin(key_genes_PMID)]

In [ ]:
gene_inv = ['HDAC2', 'DPP4', 'ACE2']
extracted_target_df_combined[extracted_target_df_combined['GENE'].isin(gene_inv)]

In [ ]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

### Functions

In [ ]:
# temp = extracted_target_df_combined.copy()

# ### check that all were capitilized in the first place
# temp[temp['GENE'] != extracted_target_df_combined['GENE']]

In [ ]:
def connect_to_genes(gene_df, ppi = protein_interaction, literature_data = extracted_target_df_combined):
    ### case fix
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    literature_data['GENE'] = literature_data['GENE'].str.upper()

    ### connect genes to those in PPI
    ### ensure only strong interactions are considered
    ppi = ppi[ppi['combined_score']>=700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    #### get literature validated genes
    literature_validated_genes = literature_data['GENE'].values
    for i,gene in tqdm(enumerate(gene_df['gene_name'])):
        validated_connected_genes = []
        PMIDs_connected_genes = []
        connected_genes = []
        if gene in ppi_available_genes:
            ### get all connected genes
            connected_genes = ppi[ppi['Translated_protein_1'] == gene]['Translated_protein_2'].tolist()
            connected_genes += ppi[ppi['Translated_protein_2'] == gene]['Translated_protein_1'].tolist()
            connected_genes = list(set(connected_genes))
            #### check which of the connected genes are literature validated
            for gene2 in connected_genes:
                if gene2 in literature_validated_genes:
                    validated_connected_genes.append(gene2)
                    ### get PMIDs for the gene
                    pmids = literature_data[literature_data['GENE'] == gene2]['PMID'].tolist()
                    PMIDs_connected_genes += pmids
        else:
            connected_genes = []
            validated_connected_genes = []
        gene_df.loc[i, 'literature_validated_connected_genes'] = ', '.join(validated_connected_genes)
        gene_df.loc[i, 'num_literature_validated_connected_genes'] = len(validated_connected_genes)
        gene_df.loc[i, 'PMIDs_connected_genes'] = ', '.join(map(str, set(PMIDs_connected_genes)))

    return gene_df

def connect_to_genes_explicit(gene_df, ppi=protein_interaction, literature_data=extracted_target_df_combined):
    """
    Expanded explicit version: For each risk gene, create a row for each neighbor (not grouped), with all details from grouped version.
    Returns a DataFrame with columns:
        risk_gene, MUT_TYPE, q_value, empirical_q_value, PMID, NUMBER_OF_ARTICLES,
        neighbor_gene, neighbor_is_lit_validated, neighbor_PMID, neighbor_NUMBER_OF_ARTICLES,
        neighbor_is_lit_validated, neighbor_PMIDs, neighbor_NUMBER_OF_ARTICLES,
    """
    ### case correction for making connections
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    literature_data['GENE'] = literature_data['GENE'].str.upper()

    ppi = ppi[ppi['combined_score'] >= 700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    literature_validated_genes = set(literature_data['GENE'].values)
    # Prepare lookup for literature data
    lit_lookup = literature_data.set_index('GENE')
    rows = []
    for idx, risk_row in gene_df.iterrows():
        risk_gene = risk_row['gene_name']
        mut_type = risk_row.get('MUT_TYPE', None)
        q_value = risk_row.get('q_value', None)
        empirical_q_value = risk_row.get('empirical_q_value', None)
        PMID = risk_row.get('PMID', None)
        NUMBER_OF_ARTICLES = risk_row.get('NUMBER_OF_ARTICLES', None)
        if risk_gene in ppi_available_genes:
            neighbors = ppi[ppi['Translated_protein_1'] == risk_gene]['Translated_protein_2'].tolist()
            neighbors += ppi[ppi['Translated_protein_2'] == risk_gene]['Translated_protein_1'].tolist()
            neighbors = list(set(neighbors))
            for neighbor in neighbors:
                neighbor_is_lit_validated = neighbor in literature_validated_genes
                neighbor_PMIDs = ''
                neighbor_NUMBER_OF_ARTICLES = None
                neighbor_PMID = None
                if neighbor_is_lit_validated and neighbor in lit_lookup.index:
                    neighbor_PMIDs = ', '.join(map(str, lit_lookup.loc[neighbor]['PMID'])) if isinstance(lit_lookup.loc[neighbor]['PMID'], (list, pd.Series)) else str(lit_lookup.loc[neighbor]['PMID'])
                    neighbor_NUMBER_OF_ARTICLES = lit_lookup.loc[neighbor]['NUMBER_OF_ARTICLES']
                    neighbor_PMID = lit_lookup.loc[neighbor]['PMID']
                rows.append({
                    'risk_gene': risk_gene,
                    'MUT_TYPE': mut_type,
                    'q_value': q_value,
                    'empirical_q_value': empirical_q_value,
                    'PMID': PMID,
                    'NUMBER_OF_ARTICLES': NUMBER_OF_ARTICLES,
                    'neighbor_gene': neighbor,
                    'neighbor_is_lit_validated': neighbor_is_lit_validated,
                    'neighbor_PMID': neighbor_PMID,
                    'neighbor_NUMBER_OF_ARTICLES': neighbor_NUMBER_OF_ARTICLES,
                    'neighbor_PMIDs': neighbor_PMIDs
                })
        else:
            rows.append({
                'risk_gene': risk_gene,
                'MUT_TYPE': mut_type,
                'q_value': q_value,
                'empirical_q_value': empirical_q_value,
                'PMID': PMID,
                'NUMBER_OF_ARTICLES': NUMBER_OF_ARTICLES,
                'neighbor_gene': None,
                'neighbor_is_lit_validated': False,
                'neighbor_PMID': None,
                'neighbor_NUMBER_OF_ARTICLES': None,
                'neighbor_PMIDs': ''
            })
    expanded_df = pd.DataFrame(rows)
    return expanded_df

## HPV+

#### Genes

In [ ]:
hpv_positive_som_genes

In [ ]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
hpv_positive_combined_genes

In [ ]:
len(set(hpv_positive_combined_genes['gene_name']))

In [ ]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [ ]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [ ]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0].reset_index(drop=True)

In [ ]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/HPV Positive validated genes.csv')

In [ ]:
hpv_positive_genes_with_lit

In [ ]:
hpv_positive_connected_genes_with_lit = connect_to_genes(hpv_positive_genes_with_lit)

In [ ]:
hpv_positive_connected_genes_with_lit

In [ ]:
all_neighbors = []
for genes in hpv_positive_connected_genes_with_lit['literature_validated_connected_genes']:
    all_neighbors.extend([g.strip() for g in genes.split(',') if g.strip()])

print(len(set(all_neighbors)))

In [ ]:
hpv_positive_connected_genes_with_lit.to_csv('Results/HPV Positive validated indirect gene results.csv')

In [ ]:
hpv_positive_connected_genes_with_lit_explicit = connect_to_genes_explicit(hpv_positive_genes_with_lit)

In [ ]:
hpv_positive_connected_genes_with_lit_explicit[hpv_positive_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]

In [ ]:
hpv_positive_connected_genes_with_lit_explicit = hpv_positive_connected_genes_with_lit_explicit[hpv_positive_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]
hpv_positive_connected_genes_with_lit_explicit.to_csv('Results/HPV Positive validated indirect gene results explicit.csv')

In [ ]:
hpv_positive_connected_genes_with_lit_explicit['neighbor_gene'].nunique()

## HPV-

#### Genes

In [ ]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()


In [ ]:
hpv_negative_combined_genes

In [ ]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [ ]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0].reset_index(drop=True)

In [ ]:
hpv_negative_gene_results_with_lit

In [ ]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/HPV Negative validated genes.csv')

In [ ]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

In [ ]:
hpv_negative_connected_genes_with_lit = connect_to_genes(hpv_negative_gene_results_with_lit)

In [ ]:
hpv_negative_connected_genes_with_lit

In [ ]:
all_neighbors = []
for genes in hpv_negative_connected_genes_with_lit['literature_validated_connected_genes']:
    all_neighbors.extend([g.strip() for g in genes.split(',') if g.strip()])

print(len(set(all_neighbors)))

In [ ]:
hpv_negative_connected_genes_with_lit.to_csv('Results/HPV Negative validated indirect gene results.csv')

In [ ]:
hpv_negative_connected_genes_with_lit_explicit = connect_to_genes_explicit(hpv_negative_connected_genes_with_lit)

In [ ]:
hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]

In [ ]:
hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]['neighbor_gene'].nunique()

In [ ]:
hpv_negative_connected_genes_with_lit_explicit = hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]
hpv_negative_connected_genes_with_lit_explicit.to_csv('Results/HPV Negative validated indirect gene results explicit.csv')

In [ ]:
print(f"Number of unique genes in hpv pos genes with lit not in hpv neg: {len(set(hpv_positive_genes_with_lit['gene_name']) - set(hpv_negative_gene_results_with_lit['gene_name']))}")
print(f"genes unique to hpv pos: {set(hpv_positive_genes_with_lit['gene_name']) - set(hpv_negative_gene_results_with_lit['gene_name'])}")
print(f"Number of unique genes in hpv neg genes with lit not in hpv pos: {len(set(hpv_negative_gene_results_with_lit['gene_name']) - set(hpv_positive_genes_with_lit['gene_name']))}")
print(f"genes unique to hpv neg: {set(hpv_negative_gene_results_with_lit['gene_name']) - set(hpv_positive_genes_with_lit['gene_name'])}")

print(f"Number of unique genes in both hpv pos and hpv neg genes with lit: {len(set(hpv_positive_genes_with_lit['gene_name']).intersection(set(hpv_negative_gene_results_with_lit['gene_name'])))}")
print(f"genes in both hpv pos and hpv neg: {set(hpv_positive_genes_with_lit['gene_name']).intersection(set(hpv_negative_gene_results_with_lit['gene_name']))}")

print(f"Number of unique genes across both hpv pos and hpv neg genes with lit: {len(set(hpv_positive_genes_with_lit['gene_name']).union(set(hpv_negative_gene_results_with_lit['gene_name'])))}")

### sanity check of the results

In [ ]:
protein_interaction[(protein_interaction['Translated_protein_1'] == 'TP53') & 
                    (protein_interaction['Translated_protein_2'] == 'THRB')]